In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import pandas as pd


def summarize_report(report_dir: str) -> pd.DataFrame:
    """
    Traverse all SNP correlation reports under {report_dir}/snp_correlation,
    extract cfDNA sample, tissue sample and correlation into a DataFrame.

    Parameters
    ----------
    report_dir : str
        Root directory that contains 'snp_correlation' subdirectory.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: ['cfdna_sample', 'tissue_sample', 'correlation'].
    """
    snp_corr_dir = os.path.join(report_dir, "snp_correlation")
    records = []

    # Compile regex patterns for robust parsing
    tissue_pattern = re.compile(r"^\s*Tissue Pileup:\s*(.+)$")
    cfdna_pattern = re.compile(r"^\s*cfDNA Pileup:\s*(.+)$")
    corr_pattern = re.compile(r"^\s*Pearson Correlation Coefficient:\s*([+-]?\d+(\.\d+)?(e[+-]?\d+)?)", re.IGNORECASE)

    # Traverse all report files in snp_correlation directory
    for fname in os.listdir(snp_corr_dir):
        if not fname.endswith("_report.txt"):
            # Skip non-report files
            continue

        report_path = os.path.join(snp_corr_dir, fname)

        tissue_sample = None
        cfdna_sample = None
        correlation = None

        # Read each report and extract the required fields
        with open(report_path, "r") as f:
            for line in f:
                # Parse tissue pileup line
                m_tissue = tissue_pattern.match(line)
                if m_tissue:
                    tissue_path = m_tissue.group(1).strip()
                    tissue_basename = os.path.basename(tissue_path)
                    # Strip "_pileup.tsv.gz" suffix to get sample name
                    tissue_sample = tissue_basename.replace("_pileup.tsv.gz", "")
                    continue

                # Parse cfDNA pileup line
                m_cfdna = cfdna_pattern.match(line)
                if m_cfdna:
                    cfdna_path = m_cfdna.group(1).strip()
                    cfdna_basename = os.path.basename(cfdna_path)
                    # Strip "_pileup.tsv.gz" suffix to get sample name
                    cfdna_sample = cfdna_basename.replace("_pileup.tsv.gz", "")
                    continue

                # Parse correlation line
                m_corr = corr_pattern.match(line)
                if m_corr:
                    correlation = float(m_corr.group(1))
                    continue

        # Only append a record if all three pieces of information are found
        if tissue_sample is not None and cfdna_sample is not None and correlation is not None:
            records.append(
                {
                    "cfdna_sample": cfdna_sample,
                    "tissue_sample": tissue_sample,
                    "correlation": correlation,
                }
            )

    # Build DataFrame from collected records
    df = pd.DataFrame(records, columns=["cfdna_sample", "tissue_sample", "correlation"])

    # Output summary TSV: {report_dir}_summary.tsv
    out_path = f"{report_dir}/summary.tsv"
    df.to_csv(out_path, sep="\t", index=False)

    return df

In [ ]:
report_dir = "../../../results/pair_qc/20251112"
summarize_report(report_dir)

In [8]:
report_dir = "../../../results/pair_qc/20251128/out/"
summarize_report(report_dir)

,cfdna_sample,tissue_sample,correlation
0,PTAY0629P13S1,PTAY0629V13S1,0.612354
1,PTAY0621P8S1,PTAY0621V8S1,0.755371
2,PTAY0615P7S1,PTAY0615V7S1,0.556715
3,PTAY0652P7H1,PTAY0652V7H1,0.268169
4,PTAY0630P7S1,PTAY0630V7S1,0.596226
5,PTAY0622P8S1,PTAY0622V8S1,0.602427
6,PTAY0656P8S1,PTAY0656V8S1,0.255528


In [9]:
report_dir = "../../../results/pair_qc/20251128/out_downsampled/"
summarize_report(report_dir)

,cfdna_sample,tissue_sample,correlation
0,PTAY0629P13S1,PTAY0629V13S1,0.670505
1,PTAY0621P8S1,PTAY0621V8S1,0.801045
2,PTAY0615P7S1,PTAY0615V7S1,0.436791
3,PTAY0652P7H1,PTAY0652V7H1,0.284202
4,PTAY0630P7S1,PTAY0630V7S1,0.678207
5,PTAY0622P8S1,PTAY0622V8S1,0.598928
6,PTAY0656P8S1,PTAY0656V8S1,0.298972
